In [84]:
%pip install ydata-profiling geopy statsmodels scipy scikit-learn pandas numpy plotly streamlit
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import sklearn.impute
import sklearn.covariance
import sklearn.model_selection
import sklearn.metrics
import sklearn.ensemble
from statsmodels.stats.outliers_influence import variance_inflation_factor
import geopy.geocoders
import geopy.distance
import geopy.extra.rate_limiter
import plotly.express as px
import streamlit as st


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [85]:
dataset= pd.read_csv('/workspaces/An-Analysis-of-Factors-Influencing-the-Success-of-World-War-II-Bombing-Missions/PROYECTO/thor_wwii_data_clean.csv')
df=[
    'bda', 'total_tons', 'tons_of_he', 'tons_of_ic', 'tons_of_frag',
    'ac_attacking', 'ac_dropping', 'altitude_feet', 'sighting_method_code',
    'tgt_type', 'tgt_priority', 'ac_lost', 'ac_damaged', 'latitude', 'longitude'
]
df_copy = dataset[df].copy()
df_copy = df_copy.dropna(subset=['bda'])
df_copy

/tmp/ipykernel_10517/3886677953.py:1: DtypeWarning: Columns (0: source_longitude, 1: time_over_target, 2: sighting_method_code, 3: bda, 4: callsign, 5: mission_comments, 6: source) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset= pd.read_csv('/workspaces/An-Analysis-of-Factors-Influencing-the-Success-of-World-War-II-Bombing-Missions/PROYECTO/thor_wwii_data_clean.csv')


,bda,total_tons,tons_of_he,tons_of_ic,tons_of_frag,ac_attacking,ac_dropping,altitude_feet,sighting_method_code,tgt_type,tgt_priority,ac_lost,ac_damaged,latitude,longitude
11862,EXCELLENT,7.70,7.70,0.0,0.00,6.0,6.0,28.0,NaN,SHWEBO AIRDROME,NaN,NaN,NaN,22.583333,95.666667
11868,GOOD,8.25,8.25,0.0,0.00,5.0,5.0,30.0,NaN,AIRPORT,NaN,NaN,NaN,22.901640,95.678936
112055,"""3 DIRECT HITS, 8 WATERLINE HITS""",3.30,3.30,0.0,0.00,2.0,2.0,NaN,NaN,JAPANESE SHIPPING,NaN,NaN,NaN,18.500000,121.616660
112056,RESULTS DOUBTFUL,2.40,2.40,0.0,0.00,1.0,1.0,NaN,NaN,JAPANESE SHIPPING,NaN,NaN,NaN,17.583333,120.333300
112057,NOT STATED,1.05,1.05,0.0,0.00,1.0,1.0,NaN,NaN,JAPANESE SHIPPING,NaN,NaN,NaN,17.583333,120.333300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141459,GOOD,0.40,0.40,0.0,0.00,4.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,24.433728,98.588649
141512,GOOD,0.40,0.40,0.0,0.00,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,24.151712,98.875415
141513,GOOD,0.20,0.20,0.0,0.00,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,24.250000,97.233333
144304,GOOD,4.68,3.60,0.0,1.08,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,24.433728,98.588649


In [86]:
metrics_eda = pd.DataFrame({
    'faltantes_%': df_copy.isnull().mean() * 100,
    'Skewness': df_copy.skew(numeric_only=True),
    'Kurtosis': df_copy.kurt(numeric_only=True)
})
metrics_eda

,faltantes_%,Skewness,Kurtosis
ac_attacking,0.000000,1.553812,4.111643
ac_damaged,100.000000,NaN,NaN
ac_dropping,2.884615,1.689076,4.862564
ac_lost,100.000000,NaN,NaN
altitude_feet,36.538462,1.386268,0.904190
bda,0.000000,NaN,NaN
latitude,0.000000,-0.886852,-0.368122
longitude,0.000000,-2.211512,3.474022
sighting_method_code,100.000000,NaN,NaN
tgt_priority,67.307692,NaN,NaN


In [87]:
bda_uniques= df_copy["bda"].unique().tolist()
print(bda_uniques)

['EXCELLENT', 'GOOD', '"3 DIRECT HITS, 8 WATERLINE HITS"', 'RESULTS DOUBTFUL', 'NOT STATED', 'COULD NOT SEE TGT ON ACCOUNT OF WX.', 'SANK ENEMY SHIP', 'HIGHLY SUCCESSFUL', '"NOT STATED, LANDED AT MINDANEO"', 'RESULTS UNCERTAIN DUE TO WX', '"1 AP SUNK, HITS ON CB"', 'NOT GIVEN', '"1 AP SUNK, 1 DAMAGED"', 'HITS ON RUNWAY AND PARKING AREA.', '"DIRECT HIT AND WATERLINE HIT ON BATTLESHIP - SET ON FIRE, ONE HIT ON SHORE.  TWO TRANSPORTS BOMBED AND MISSED"', 'ENEMY AIRCRAFT CARRIER. NO HITS OBSERVED', 'RESULTS UNCERTAIN', '"1 DESTROYER SUNK, 3 DIRECT HITS ON BATTLESHIP, SEVERAL SMALLER VESSELS BELIEVED HIT."', 'NO RESULTS', 'RESULTS UNKNOWN', 'NO HITS OBSERVED', '"ATTACKED TRANSPORTS AND DESTROYER, RESULTS UNKNOWN"', '"SEVEN SHIPS TARGETED, RESULTS UNCERTAIN DUE TO LIGHT OVERCAST"', 'DOCKS WERE HIT.  LARGE FIRE BURNING BEFORE AND AFTER ATTACK.', '4 SHIPS IN GULF TARGETED.  RESULTS UNCERTAINDUE TO CLOUDS', '"HITS ON CRUISER AND TRANSPORT, BOTH LEFT BURNING"', '3 LARGE FIRES', 'HIT SCORED ON AU

In [88]:
df_copy["bda"].unique().tolist()

['EXCELLENT',
 'GOOD',
 '"3 DIRECT HITS, 8 WATERLINE HITS"',
 'RESULTS DOUBTFUL',
 'NOT STATED',
 'COULD NOT SEE TGT ON ACCOUNT OF WX.',
 'SANK ENEMY SHIP',
 'HIGHLY SUCCESSFUL',
 '"NOT STATED, LANDED AT MINDANEO"',
 'RESULTS UNCERTAIN DUE TO WX',
 '"1 AP SUNK, HITS ON CB"',
 'NOT GIVEN',
 '"1 AP SUNK, 1 DAMAGED"',
 'HITS ON RUNWAY AND PARKING AREA.',
 '"DIRECT HIT AND WATERLINE HIT ON BATTLESHIP - SET ON FIRE, ONE HIT ON SHORE.  TWO TRANSPORTS BOMBED AND MISSED"',
 'ENEMY AIRCRAFT CARRIER. NO HITS OBSERVED',
 'RESULTS UNCERTAIN',
 '"1 DESTROYER SUNK, 3 DIRECT HITS ON BATTLESHIP, SEVERAL SMALLER VESSELS BELIEVED HIT."',
 'NO RESULTS',
 'RESULTS UNKNOWN',
 'NO HITS OBSERVED',
 '"ATTACKED TRANSPORTS AND DESTROYER, RESULTS UNKNOWN"',
 '"SEVEN SHIPS TARGETED, RESULTS UNCERTAIN DUE TO LIGHT OVERCAST"',
 'DOCKS WERE HIT.  LARGE FIRE BURNING BEFORE AND AFTER ATTACK.',
 '4 SHIPS IN GULF TARGETED.  RESULTS UNCERTAINDUE TO CLOUDS',
 '"HITS ON CRUISER AND TRANSPORT, BOTH LEFT BURNING"',
 '3 LARGE

In [89]:
map_bda = {
    'EXCELLENT': 1, 'GOOD': 1, 'HIGHLY SUCCESSFUL': 1, 'FAIR': 1,
    '3 DIRECT HITS, 8 WATERLINE HITS': 1, '2ND RUN WATERLINE HIT': 1,
    'HIT PORT BOW': 1, 'HIT STERN OF CV': 1, 'HIT ON STERN PATTERN': 1,
    'SANK ENEMY SHIP': 1, '1 DD SANK': 1, '1 AP SUNK, HITS ON CB': 1,
    '1 AP SUNK, 1 DAMAGED': 1, '1 TRANSPORT SEEN TO CAPSIZE FROM BOMBING': 1,
    '1 DIRECT HIT ON BATTLESHIP, 2 NEAR MISSES. TRANSPORT SUNK.': 1,
    'HITS ON RUNWAY AND PARKING AREA.': 1, 'DAMAGED RUNWAYS AND PARKED AIRCRAFT': 1,
    'DAMAGED RUNWAYS AND SEVERAL BUILDINGS': 1, 'FURTHER DAMAGE TO AIRFIELD': 1,
    'DIRECT HIT ON FUEL DUMP': 1, '3 LARGE FIRES': 1, 'ONE TRANSPORT BURNING AND SINKING': 1,
    'CA SMOKING HEAVILY': 1, 'CRUISER HIT, SMOKING BADLY': 1, '1 BB OR CA AFIRE.  1 TRANS IN FLAMES': 1,
    'HITS ON CRUISER AND TRANSPORT, BOTH LEFT BURNING': 1, 'HIT SCORED ON AUX VESSEL AND CRUISER': 1,
    'HITS ON HANGER AND PARKED PLANES, ESTIMATED 40 PLANES DESTROYED, 25  DAMAGED': 1,
    'AIRPORT HIT, AT LEAST ONE ENEMY PLANE DESTROYED ON GROUND': 1, 'BRACKETED ONE OF FOUR BOATS IN HARBOR, BELIEVE DAMAGE DONE.': 1,
    'DOCKS WERE HIT.  LARGE FIRE BURNING BEFORE AND AFTER ATTACK.': 1,
    '2 TRANSPORTS AND 2 BARGES HIT.  PLANES HIT BY LIGHT AA AND BOMB FRAGMENTS OF OWN WEAPONS': 1,
    '4 BOMBS MISSED ENEMY CRUISER, 3 BOMBS STRUCK TRANSPORT NEAR SHORE - DIRECT HITS, TRANSPORT BURNING': 1,
    'DIRECT HIT AND WATERLINE HIT ON BATTLESHIP - SET ON FIRE, ONE HIT ON SHORE.  TWO TRANSPORTS BOMBED AND MISSED': 1,
    '1 DESTROYER SUNK, 3 DIRECT HITS ON BATTLESHIP, SEVERAL SMALLER VESSELS BELIEVED HIT.': 1,
    'NO HITS': 0, 'NO HITS OBSERVED': 0, 'NO RESULTS': 0, 'ENEMY AIRCRAFT CARRIER. NO HITS OBSERVED': 0,
    'BOMBS HIT CLOSE TO CRUISER, BUT NO DIRECT HITS': 0, 'BOMB RACKS FAULTY': 0, 'MALFUNCTION OF RACKS': 0,
    '(3 DUDS)': 0, 'NEAR STERN MISSES': 0, 'BDA NOT PERFORMED DUE TO ENEMY SEARCHLIGHTS AND AA FIRE': 0,
    'BDA NOT PERFORMED DUE TO ENEMY INTERCEPTORS': 0, 'GROUP CO OF 7TH BOMB GROUP SHOT DOWN': 0,
    'NOT STATED': np.nan, 'NOT GIVEN': np.nan, 'NOT STATED, LANDED AT MINDANEO': np.nan,
    'RESULTS DOUBTFUL': np.nan, 'DOUBTFUL': np.nan, 'RESULTS UNCERTAIN': np.nan, 'RESULTS UNKNOWN': np.nan,
    'ATTACKED TRANSPORTS AND DESTROYER, RESULTS UNKNOWN': np.nan, 'COULD NOT SEE TGT ON ACCOUNT OF WX.': np.nan,
    'RESULTS UNCERTAIN DUE TO WX': np.nan, 'WEATHER TOO BAD TO LOCATE TGT': np.nan,
    'SEVEN SHIPS TARGETED, RESULTS UNCERTAIN DUE TO LIGHT OVERCAST': np.nan,
    '4 SHIPS IN GULF TARGETED.  RESULTS UNCERTAINDUE TO CLOUDS': np.nan,
    '1 BOAT BELIEVED HIT, UNABLE TO OBSERVE OTHER BOMB RUNS DUE TO CLOUDS': np.nan,
    'MADE 3 RUNS': np.nan, 'CV SLOWED DOWN': np.nan
}

df_copy['bda_new'] = df_copy['bda'].astype(str).str.replace('"', '').str.strip()
df_copy['victory'] = df_copy['bda_new'].map(map_bda)
df_copy = df_copy.drop(columns=['bda_new'])
df_copy = df_copy.dropna(subset=['victory'])
df_copy[["bda","victory"]]


,bda,victory
11862,EXCELLENT,1.0
11868,GOOD,1.0
112055,"""3 DIRECT HITS, 8 WATERLINE HITS""",1.0
112089,SANK ENEMY SHIP,1.0
112092,HIGHLY SUCCESSFUL,1.0
...,...,...
141459,GOOD,1.0
141512,GOOD,1.0
141513,GOOD,1.0
144304,GOOD,1.0


In [90]:
df_copy = df_copy.drop(columns=['bda'])
# cuantos aviones soltaron bombas frente a los planeados?
df_copy['ef_plane_rate'] = df_copy['ac_dropping'] / df_copy['ac_attacking']
df_copy['tons_per_atk_plane'] = df_copy['total_tons'] / df_copy['ac_attacking']
estadisticas_daño = df_copy[['total_tons', 'ef_plane_rate', 'altitude_feet']].agg(
    ['mean', 'median', 'std', 'skew', 'kurt'])
estadisticas_daño

,total_tons,ef_plane_rate,altitude_feet
mean,4.023494,0.985944,4364.078431
median,3.220000,1.000000,0.000000
std,3.247778,0.083149,6697.754140
skew,1.463098,-6.821645,1.472408
kurt,2.975377,49.679649,1.179094


In [ ]:
for col in ['total_tons', 'altitude_feet']:
    datos_validos = df_copy[col].dropna()
    stat, p_val = stats.kstest(datos_validos, 'norm', args=(datos_validos.mean(), datos_validos.std()))
    print(f"Variable {col} -> p-valor Kolmogorov-Smirnov: {p_val:.4f}")

Variable total_tons -> p-valor KS: 0.2041
Variable altitude_feet -> p-valor KS: 0.0000


In [92]:
print("--- MATRIZ DE CORRELACIÓN BIVARIADA (MÉTODO SPEARMAN) ---")
#Spearman porque altitude_feet no es normal y mide relaciones no lineales
variables_numericas = ['total_tons', 'altitude_feet', 'ac_attacking', 'ac_dropping']
matriz_correlacion = df_copy[variables_numericas].corr(method='spearman')
print(matriz_correlacion)

print("\n--- SIGNIFICANCIA ESTADÍSTICA DE LA CORRELACIÓN")
# Toda correlación necesita p-valor
df_filtered = df_copy[['total_tons', 'altitude_feet']].dropna()
coeficiente, p_valor_corr = stats.spearmanr(df_filtered['total_tons'], df_filtered['altitude_feet'])
print(f"Coeficiente de Spearman: {coeficiente:.4f}")
print(f"p-valor de la correlación: {p_valor_corr:.4f} ({'Significativa' if p_valor_corr < 0.05 else 'No significativa'})")
#### debido a que ac_attacking y ac_dropping tienen multicolinealidad casi perfecta se decide eliminar ac_attacking debido a que no representa al 100% el daño ejecutado
#### coeficiente de spearman para toneladas vs altitud queda descartado gracias al p-valor el cual indica que lo mas probable es que sea pura casualidad.

--- MATRIZ DE CORRELACIÓN BIVARIADA (MÉTODO SPEARMAN) ---
               total_tons  altitude_feet  ac_attacking  ac_dropping
total_tons       1.000000      -0.190586      0.477564     0.511097
altitude_feet   -0.190586       1.000000     -0.015801    -0.015801
ac_attacking     0.477564      -0.015801      1.000000     0.965688
ac_dropping      0.511097      -0.015801      0.965688     1.000000

--- SIGNIFICANCIA ESTADÍSTICA DE LA CORRELACIÓN
Coeficiente de Spearman: -0.1906
p-valor de la correlación: 0.1994 (No significativa)


In [93]:
columnas_numericas = ['total_tons', 'altitude_feet', 'ac_dropping']
df_antes = df_copy[columnas_numericas].copy()

imputador_knn = sklearn.impute.KNNImputer(n_neighbors=5)
df_imputado = pd.DataFrame(
    imputador_knn.fit_transform(df_antes),
    columns=columnas_numericas,
    index=df_copy.index
)

#EVIDENCIA EMPÍRICA: Comparar estadísticas antes vs después
print("--- EVALUACIÓN DEL IMPACTO DE LA IMPUTACIÓN ---")
for col in columnas_numericas:
    print(f"\nVariable: {col}")
    print(f"  Media ANTES: {df_antes[col].mean():.4f} | Media DESPUÉS: {df_imputado[col].mean():.4f}")
    print(f"  Desviación ANTES: {df_antes[col].std():.4f} | Desviación DESPUÉS: {df_imputado[col].std():.4f}")

    # Prueba de Kolmogorov-Smirnov para confirmar si la imputación alteró la distribución original
    stat, p_val_ks = stats.ks_2samp(df_antes[col].dropna(), df_imputado[col])
    print(f"  Prueba KS p-valor: {p_val_ks:.4f} -> "
          f"{'Estructura Preservada (Excelente)' if p_val_ks > 0.05 else 'Distribución Alterada'}")

df_copy[columnas_numericas] = df_imputado


--- EVALUACIÓN DEL IMPACTO DE LA IMPUTACIÓN ---

Variable: total_tons
  Media ANTES: 4.0235 | Media DESPUÉS: 4.0039
  Desviación ANTES: 3.2478 | Desviación DESPUÉS: 3.1694
  Prueba KS p-valor: 1.0000 -> Estructura Preservada (Excelente)

Variable: altitude_feet
  Media ANTES: 4364.0784 | Media DESPUÉS: 5088.7562
  Desviación ANTES: 6697.7541 | Desviación DESPUÉS: 5433.7621
  Prueba KS p-valor: 0.0394 -> Distribución Alterada

Variable: ac_dropping
  Media ANTES: 4.3133 | Media DESPUÉS: 4.3190
  Desviación ANTES: 2.7227 | Desviación DESPUÉS: 2.6906
  Prueba KS p-valor: 1.0000 -> Estructura Preservada (Excelente)


In [ ]:
print("DETECCIÓN DE OUTLIERS ")
z_scores = np.abs(stats.zscore(df_copy['total_tons']))
outliers_z = z_scores > 3
print(f"Outliers univariados detectados con Z-score (total_tons): {outliers_z.sum()}")

#IQR para 'altitude_feet' (Variable No Normal, umbral de 1.5 * IQR)
q1 = df_copy['altitude_feet'].quantile(0.25)
q3 = df_copy['altitude_feet'].quantile(0.75)
iqr = q3 - q1
outliers_iqr = (df_copy['altitude_feet'] < (q1 - 1.5 * iqr)) | (df_copy['altitude_feet'] > (q3 + 1.5 * iqr))
print(f"Outliers univariados detectados con IQR (altitude_feet): {outliers_iqr.sum()}")


# --- 2. DETECCIÓN MULTIVARIADA (DISTANCIA DE MAHALANOBIS) ---

# Columnas numéricas.
columnas_outliers = ['total_tons', 'altitude_feet', 'ac_dropping']

mcd = sklearn.covariance.MinCovDet()
mcd.fit(df_copy[columnas_outliers])
distancias_mahalanobis = mcd.mahalanobis(df_copy[columnas_outliers])

#umbral crítico basado en distribución chi-cuadrado (confianza al 99%)
grados_libertad = len(columnas_outliers)
umbral_chi2 = stats.chi2.ppf(0.99, grados_libertad)

outliers_mahalanobis = distancias_mahalanobis > umbral_chi2
print(f"Umbral chi-Cuadrado: {umbral_chi2:.4f}")
print(f"Outliers multivariados detectados con Mahalanobis: {outliers_mahalanobis.sum()}")


# --- 3. EVALUACIÓN DE IMPACTO Y TRATAMIENTO ---

media_tons_antes = df_copy['total_tons'].mean()
media_alt_antes = df_copy['altitude_feet'].mean()


# Eliminar los outliers multivariados del dataset
df_limpio = df_copy[~outliers_mahalanobis].copy()

print("\n--- IMPACTO EN LOS ESTADÍSTICOS TRAS EL TRATAMIENTO ---")
print(f"Registros eliminados: {outliers_mahalanobis.sum()} ({outliers_mahalanobis.mean()*100:.2f}% del dataset)")
print(f"Media total_tons -> ANTES: {media_tons_antes:.2f} | DESPUÉS: {df_limpio['total_tons'].mean():.2f}")
print(f"Media altitude_feet -> ANTES: {media_alt_antes:.2f} | DESPUÉS: {df_limpio['altitude_feet'].mean():.2f}")
df_copy = df_limpio
df_copy


DETECCIÓN DE OUTLIERS 
Outliers univariados detectados con Z-score (total_tons): 1
Outliers univariados detectados con IQR (altitude_feet): 2


Umbral chi-Cuadrado: 11.3449
Outliers multivariados detectados con Mahalanobis: 19

--- IMPACTO EN LOS ESTADÍSTICOS TRAS EL TRATAMIENTO ---
Registros eliminados: 19 (22.35% del dataset)
Media total_tons -> ANTES: 4.00 | DESPUÉS: 3.30
Media altitude_feet -> ANTES: 5088.76 | DESPUÉS: 4335.16


,total_tons,tons_of_he,tons_of_ic,tons_of_frag,ac_attacking,ac_dropping,altitude_feet,sighting_method_code,tgt_type,tgt_priority,ac_lost,ac_damaged,latitude,longitude,victory,ef_plane_rate,tons_per_atk_plane
11862,7.700,7.7,0.0,0.00,6.0,6.0,28.0,NaN,SHWEBO AIRDROME,NaN,NaN,NaN,22.583333,95.666667,1.0,1.0,1.283333
112055,3.300,3.3,0.0,0.00,2.0,2.0,7900.0,NaN,JAPANESE SHIPPING,NaN,NaN,NaN,18.500000,121.616660,1.0,1.0,1.650000
112089,1.038,NaN,0.0,0.00,1.0,1.0,0.0,NaN,JAPANESE SHIPPING,NaN,NaN,NaN,17.583333,120.333300,1.0,1.0,NaN
112092,2.400,2.4,0.0,0.00,2.0,2.0,6200.0,NaN,JAPANESE SHIPPING,NaN,NaN,NaN,14.591588,120.977189,1.0,1.0,1.200000
112367,1.038,NaN,0.0,0.00,1.0,1.0,0.0,NaN,DOCKS,NaN,NaN,NaN,7.076400,125.612900,1.0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141459,0.400,0.4,0.0,0.00,4.0,4.0,4801.0,NaN,NaN,NaN,NaN,NaN,24.433728,98.588649,1.0,1.0,0.100000
141512,0.400,0.4,0.0,0.00,2.0,2.0,1400.0,NaN,NaN,NaN,NaN,NaN,24.151712,98.875415,1.0,1.0,0.200000
141513,0.200,0.2,0.0,0.00,2.0,2.0,1400.0,NaN,NaN,NaN,NaN,NaN,24.250000,97.233333,1.0,1.0,0.100000
144304,4.680,3.6,0.0,1.08,6.0,6.0,6000.0,NaN,NaN,NaN,NaN,NaN,24.433728,98.588649,1.0,1.0,0.780000


In [ ]:
print(" 1. DIAGNÓSTICO DE MULTICOLINEALIDAD INICIAL")

from statsmodels.stats.outliers_influence import variance_inflation_factor

predictores_candidatos = ['total_tons', 'altitude_feet', 'ac_attacking', 'ac_dropping',"ef_plane_rate","tons_per_atk_plane"]

# Creo un dataframe con los predictores candidatos y agregar una columna constante
df_vif_antes = df_limpio[predictores_candidatos].copy()
df_vif_antes['const'] = 1.0
# Reemplazar infinitos y valores NaN
df_vif_antes.replace([np.inf, -np.inf], np.nan, inplace=True)
df_vif_antes_cleaned = df_vif_antes.dropna()

vif_antes = pd.DataFrame()
vif_antes["Variable"] = predictores_candidatos
vif_antes["VIF_Antes"] = [
    variance_inflation_factor(df_vif_antes_cleaned.values, i)
    for i in range(len(predictores_candidatos))
]
print(vif_antes)

print("\n=== 2. APLICACIÓN DE DECISIÓN (DESCARTE) ===")
# Como ac_attacking y ac_dropping tienen un VIF grande entonces duplican información,
# eliminamos ac_attacking y nos quedamos con ac_dropping (aviones que ejecutaron el daño real), al igual que tons per attacking plane.
predictores_finales = ['total_tons', 'altitude_feet', 'ac_dropping',"ef_plane_rate"]
df_vif_despues = df_limpio[predictores_finales].copy()
df_vif_despues['const'] = 1.0
df_vif_despues.replace([np.inf, -np.inf], np.nan, inplace=True)
df_vif_despues_cleaned = df_vif_despues.dropna()

vif_despues = pd.DataFrame()
vif_despues["Variable"] = predictores_finales
vif_despues["VIF_Final"] = [
    variance_inflation_factor(df_vif_despues_cleaned.values, i)
    for i in range(len(predictores_finales))
]
print(vif_despues)

 1. DIAGNÓSTICO DE MULTICOLINEALIDAD INICIAL
             Variable  VIF_Antes
0          total_tons   7.184054
1       altitude_feet   1.111980
2        ac_attacking        inf
3         ac_dropping        inf
4       ef_plane_rate        inf
5  tons_per_atk_plane   4.716722

=== 2. APLICACIÓN DE DECISIÓN (DESCARTE) ===
        Variable  VIF_Final
0     total_tons   1.761194
1  altitude_feet   1.066140
2    ac_dropping   1.752181
3  ef_plane_rate   1.026543


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [96]:
df_copy = df_copy.drop(columns=['ac_attacking'])


In [97]:
print("\n=== 3. PRUEBAS DE ASOCIACIÓN FORMALES CONTRA EL TARGET ===")

for i in predictores_finales:
    grupo_victoria = df_copy[df_copy['victory'] == 1.0][i]
    grupo_derrota = df_copy[df_copy['victory'] == 0.0][i]

    # Prueba U de Mann-Whitney
    stat, p_valor = stats.mannwhitneyu(grupo_victoria, grupo_derrota)

    print(f"\nVariable: {i}")
    print(f"  Mediana en Victorias: {grupo_victoria.median():.2f} | Mediana en Derrotas: {grupo_derrota.median():.2f}")
    print(f"  U-Test p-valor: {p_valor:.4f}")
    if p_valor < 0.05:
        print(f"  RESULTADO: SÍ ayuda a predecir. Hay una diferencia estadísticamente significativa.")
    else:
        print(f"  RESULTADO: NO ayuda a predecir de forma aislada (No significativa).")



=== 3. PRUEBAS DE ASOCIACIÓN FORMALES CONTRA EL TARGET ===

Variable: total_tons
  Mediana en Victorias: 3.00 | Mediana en Derrotas: 4.65
  U-Test p-valor: 0.5075
  RESULTADO: NO ayuda a predecir de forma aislada (No significativa).

Variable: altitude_feet
  Mediana en Victorias: 4801.00 | Mediana en Derrotas: 0.00
  U-Test p-valor: 0.5986
  RESULTADO: NO ayuda a predecir de forma aislada (No significativa).

Variable: ac_dropping
  Mediana en Victorias: 4.00 | Mediana en Derrotas: 3.00
  U-Test p-valor: 1.0000
  RESULTADO: NO ayuda a predecir de forma aislada (No significativa).

Variable: ef_plane_rate
  Mediana en Victorias: 1.00 | Mediana en Derrotas: 1.00
  U-Test p-valor: nan
  RESULTADO: NO ayuda a predecir de forma aislada (No significativa).
